In [27]:
from bs4 import BeautifulSoup
import requests
import re
import json
import os
import urllib.request
import shutil
import math
import pandas as pd

#######################################
#         File EXTRACTOOOOOOOR        #
#######################################

# Define target website
URL = "https://www.ipos.gov.sg/about-ip/form-fees"
# Define the domain
domain = "https://www.ipos.gov.sg"

# Define variables
files = {}
folder_path = "docs"
hrefSchema = {}
whitelistedFileType = ["pdf", "xls", "xlsx", "csv", "tsv"]

# Define custom file path after /files (e.g. "/folder1/folder2")
# Output -> "/files/folder1/folder2/document.pdf"
docsFolderPath = "/manage-ip/legal-decision"

# Define the folder path
file_name = 'docs'
file_name_zip = file_name+'.zip'

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Connection': 'keep-alive',
}

# Remove existing folders and/or files if exists from previous session
try:
    while os.path.exists(file_name) or os.path.exists(file_name_zip) is True:
        shutil.rmtree(file_name)
        os.remove(file_name_zip)
except:
  pass

# Generate dictionary
def getDictionary(link):
  count = 0
  URL = f"{link}"

  # Pulls entire HTML code and parse it through BeautifulSoup
  page = requests.get(URL, headers=headers)
  soup = BeautifulSoup(page.content, "html.parser")

  # Loop through all anchor tags to find downloadable files
  for i in soup.find_all(['a']):
    href = i.get('href')
    # Check if the link is a document link
    try:
        if href and '/docs/' in i['href'] or href and 'go.gov.sg' in i['href']:
            # Build dictionary
            if i['href'][:i['href'].find('?')].split("/")[-1].split(".")[1] in whitelistedFileType:
                files[count] = {"Title": i.text, "File name": i['href'][:i['href'].find('?')].split("/")[-1], "Download Link": i['href'] if 'go.gov.sg' in i['href'] else domain + i['href'] if domain not in i['href'] else i['href'], "File path": f"/files{docsFolderPath if True else None}/{i['href'][:i['href'].find('?')].split("/")[-1]}", "Updated link": i['href'][:i['href'].find('?')], "Original link": i['href']}
                r = requests.head(files[count]["Download Link"], allow_redirects=True)
                # Resolve potential redirects
                files[count]["Download Link"] = r.url.split('%')[0]
                files[count]["File name"] = r.url[:r.url.find('?')].split("/")[-1] if '?' in r.url else r.url.split("/")[-1]
                files[count]["File name"] = files[count]["File name"].split('%')[0]
                
                # print(files[count]["Download Link"], files[count]["File name"])
                count += 1
            else:
                print("Blacklisted file type:", i['href'][:i['href'].find('?')].split("/")[-1].split(".")[1], domain + href if "https://" not in href else href)
    except:
        print("Error:", href)
    
  downloadFiles()

# Download the files based on the constructed dictionary
def downloadFiles():
  # Create the output folder if it doesn't exist
  if not os.path.exists(folder_path):
      os.makedirs(folder_path)

  # Configure request headers for downloading
  headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Connection': 'keep-alive',
  }
  opener = urllib.request.build_opener()
  opener.addheaders = [(key, value) for key, value in headers.items()]
  urllib.request.install_opener(opener)

  # Iterate through each file and download it
  for index, value in enumerate(files):
    try:
        # Stores each dictionary item's download link
        url = files[index]['Download Link']

        # Define file path and name
        file_path = os.path.join(f"{folder_path}", f"{files[index]['File name']}")
        
        # Download file
        urllib.request.urlretrieve(url, file_path)

    except:
        print("Error:", url)

  # Generate file size metadata and print the JSON schema
  for index, value in enumerate(files):
    try:
        # Debating between dividing 1024 or 1000. Generally 1 Kilo = 1000(grams), But in Binary 1 Kilo = 1024 bytes
        fileSize = str(math.ceil(os.path.getsize(f"docs/{files[index]['File name']}")/1024))
        # Calculate and define file size
        fileSize = fileSize + ' KB' if len(fileSize) <= 3 else str(round((int(fileSize)/1000), 2)) + ' MB' if len(fileSize) >= 4 and len(fileSize) < 7 else str(fileSize) + ' B'    
        
        hrefSchema = {
              "type": "text",
              "marks": [
                {
                  "type": "link",
                  "attrs": {
                    "href": f"/files{docsFolderPath if True else None}/{files[index]['File name']}"
                    }
                }
              ],
              # Original File Name is used as we want to display Circular 1 instead of circular-1
              "text": f"{files[index]['Title']} [{'DOCX' if '.docx' in files[index]['Download Link'] else 'DOC' if '.doc' in files[index]['Download Link'] else 'XLXS' if '.xlxs' in files[index]['Download Link'] else 'XLS' if '.xls' in files[index]['Download Link'] else 'ZIP' if '.zip' in files[index]['Download Link'] else 'PDF'}, {fileSize}]"
            }
        print(json.dumps(hrefSchema))
    except Exception as e:
      pass

getDictionary(URL)

# Export report to .csv
df = pd.DataFrame.from_dict(files, orient='index')
df.to_csv("file-extractor-report.csv", index=False)

Blacklisted file type: docx https://www.ipos.gov.sg/docs/default-source/resources-library/trade-marks/resources/request-for-acceleration-of-tm-applicationsbcea1b77c2d0635fa1cdff0000abd271.docx?sfvrsn=8b3a7859_4
Blacklisted file type: doc https://www.ipos.gov.sg/docs/default-source/resources-library/trade-marks/trade-mark-forms/form-tm4---otc67cd1c77c2d0635fa1cdff0000abd271.doc?sfvrsn=131d7f59_4
Blacklisted file type: doc https://www.ipos.gov.sg/docs/default-source/resources-library/form-cs1---otc.doc?sfvrsn=9ca77b59_2
Blacklisted file type: doc https://www.ipos.gov.sg/docs/default-source/default-document-library/form-tm8---otc.doc?sfvrsn=6f1f7f59_0
Blacklisted file type: doc https://www.ipos.gov.sg/docs/default-source/resources-library/form-cs4---otc.doc?sfvrsn=84a77b59_2
Blacklisted file type: doc https://www.ipos.gov.sg/docs/default-source/resources-library/trade-marks/trade-mark-forms/form-tm10---otc3d402177c2d0635fa1cdff0000abd271.doc?sfvrsn=904259_2
Blacklisted file type: doc http

In [ ]:
from bs4 import BeautifulSoup
import requests
import re
import json
import os
import urllib.request
import shutil
import math
import pandas as pd

#######################################
#         Image EXTRACTOOOOOOOR       #
#######################################

# Define target website
URL = "https://www.worldcitiessummit.com.sg/partners-n-media/wcs-2024-sponsors"
# Define the domain
domain = "https://www.worldcitiessummit.com.sg"
# Define staging site for redirections report
staging = "https://staging.d361y72dz9pmyk.amplifyapp.com/"

# Define variables
files = {}
folder_path = "images"
hrefSchema = {}

# Define custom file path after /images (e.g. "/folder1/folder2")
# Output -> "/images/folder1/folder2/document.png"
docsFolderPath = ""

# Define the folder path
file_name = 'docs'
file_name_zip = file_name+'.zip'

# Remove existing folders and/or files if exists from previous session
try:
    while os.path.exists(file_name) or os.path.exists(file_name_zip) is True:
        shutil.rmtree(file_name)
        os.remove(file_name_zip)
except:
  pass

def getDictionary(link):
  count = 0
  URL = f"{link}"

  # Pulls entire HTML code and parse it through BeautifulSoup
  page = requests.get(URL)
  # page = """<div class="fancybox-thumbs"><ul><li data-index="0" tabindex="0" class="fancybox-thumbs-active"><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/1.jpg" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/1.jpg" style="width: 133px; height: 75px; margin-top: 0px; margin-left: -15px;"></li><li data-index="1" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/17.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/17.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="2" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/7.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/7.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="3" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/5.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/5.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="4" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/4.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/4.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="5" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/27.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/27.png" style="width: 133px; height: 75px; margin-top: 0px; margin-left: -15px;"></li><li data-index="6" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/12.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/12.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="7" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/26.jpg" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/26.jpg" style="width: 133px; height: 75px; margin-top: 0px; margin-left: -15px;"></li><li data-index="8" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/8.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/8.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="9" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/6.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/6.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="10" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/23.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/23.png" style="width: 166px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="11" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/29.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/29.png" style="width: 133px; height: 75px; margin-top: 0px; margin-left: -15px;"></li><li data-index="12" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/13.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/13.png" style="width: 164px; height: 75px; margin-top: 0px; margin-left: -30px;"></li><li data-index="13" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/20.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/20.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="14" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/10.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/10.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="15" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/11.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/11.png" style="width: 164px; height: 75px; margin-top: 0px; margin-left: -30px;"></li><li data-index="16" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/16.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/16.png" style="width: 164px; height: 75px; margin-top: 0px; margin-left: -30px;"></li><li data-index="17" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/3.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/3.png" style="width: 164px; height: 75px; margin-top: 0px; margin-left: -30px;"></li><li data-index="18" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/9.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/9.png" style="width: 166px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="19" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/25.jpg" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/25.jpg" style="width: 133px; height: 75px; margin-top: 0px; margin-left: -15px;"></li><li data-index="20" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/28.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/28.png" style="width: 133px; height: 75px; margin-top: 0px; margin-left: -15px;"></li><li data-index="21" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/19.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/19.png" style="width: 166px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="22" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/24.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/24.png" style="width: 133px; height: 75px; margin-top: 0px; margin-left: -15px;"></li><li data-index="23" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/21.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/21.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="24" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/2.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/2.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="25" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/15.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/15.png" style="width: 164px; height: 75px; margin-top: 0px; margin-left: -30px;"></li><li data-index="26" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/18.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/18.png" style="width: 166px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="27" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/14.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/14.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="28" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/22.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/22.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li></ul></div>"""
  soup = BeautifulSoup(page.content, "html.parser")

  # Loop through all anchor tags to find downloadable images
  for i in soup.find_all(['img']):
    try:    
        # Build dictionary
        files[count] = {"Updated image name": i['src'].split("/")[-1], "Download link": domain + i['src'].replace(" ", "%20"), "Staging link": staging + f"images/{docsFolderPath if True else None}{i['src'][:i['src'].find('?')].split("/")[-1]}"}
        print(files[count]["Download link"], files[count]["Updated image name"])
        count += 1
        
    except Exception as e:
        pass

  downloadFiles()

# Download the images based on the constructed dictionary
def downloadFiles():
  # Create the output folder if it doesn't exist
  if not os.path.exists(folder_path):
      os.makedirs(folder_path)
  # Configure request headers for downloading
  headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Connection': 'keep-alive',
  }
  opener = urllib.request.build_opener()
  opener.addheaders = [(key, value) for key, value in headers.items()]
  urllib.request.install_opener(opener)

  # Iterate through each file and download it
  for index, value in enumerate(files):
    try:
        # Stores each dictionary item's download link
        url = files[index]['Download link']

        # Define file path and name
        file_path = os.path.join(f"{folder_path}", f"{files[index]['Updated image name']}")
        
        # Download image
        response = requests.get(url, headers=headers, stream=True)
        if response.status_code == 200:
            with open(file_path, "wb") as f:
                for chunk in response.iter_content(1024):
                    f.write(chunk)
        
        hrefSchema = {
            "type": "image",
            "src": f"/images{docsFolderPath if True else None}/{files[index]['Updated image name']}",
            "alt": ""
        }
        print(json.dumps(hrefSchema))
    except Exception as e:
        pass

getDictionary(URL)

# Export report to .csv
df = pd.DataFrame.from_dict(files, orient='index')
df.to_csv("image-extractor-report.csv", index=False)

In [ ]:
from bs4 import BeautifulSoup
import requests
import re
import json
import os
import urllib.request
import shutil
import math
import pandas as pd

#######################################
#     Link Extractooooor (Website)    #
#######################################

# Define target website
url = "https://www.healthprofessionals.gov.sg/smc/announcements"
domain = "https://www.healthprofessionals.gov.sg"
files = {}
count = 0

# Pulls entire HTML code and parse it through BeautifulSoup
page = requests.get(url)
soup = BeautifulSoup(page.content, "html.parser")
soup = soup.find("div", class_="col-sm-12 announcement-all")

# Loop through all anchor tags to find links
for i, v in enumerate(soup.find_all(['a'])):
    print(v.text)
    # Build dictionary for easier search
    files[count] = {"Title": v.text, "Link": domain + v['href']}
    count += 1
    
# Export report to .csv
df = pd.DataFrame.from_dict(files, orient='index')
df.to_csv("links-report.csv", index=False)